# Ocean-Wind-Currents Simulation Demo

This notebook demonstrates the key features of the ocean-wind-currents-simulation package:

1. Data acquisition from Copernicus and HYCOM
2. Wind and current analysis
3. Visualization with quiver plots
4. Particle tracking simulation
5. Animation creation

## Setup and Imports

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import our modules
from src.data.copernicus_client import CopernicusClient
from src.data.hycom_client import HYCOMClient
from src.analysis.wind_analysis import WindAnalyzer
from src.analysis.current_analysis import CurrentAnalyzer
from src.visualization.quiver_plots import QuiverPlotter
from src.visualization.animations import FieldAnimator
from src.simulation.particle_tracker import ParticleTracker

print('✓ All modules imported successfully!')

## 1. Data Acquisition

### 1.1 Initialize Clients

Note: This demo uses mock data. For real data, provide your credentials.

In [ ]:
# Initialize Copernicus client
copernicus = CopernicusClient()

# Initialize HYCOM client
hycom = HYCOMClient(product='GLBv0.08/expt_93.0')

print('✓ Clients initialized')

### 1.2 Download Wind Data

In [ ]:
# Define region (Gulf of Mexico)
lat_min, lat_max = 20.0, 30.0
lon_min, lon_max = -95.0, -85.0
start_date = '2024-01-01'
end_date = '2024-01-03'

# Download wind data
wind_data = copernicus.get_wind_data(
    lat_min, lat_max, lon_min, lon_max,
    start_date, end_date
)

print(f'Wind data shape: {wind_data.dims}')
print(f'Variables: {list(wind_data.data_vars)}')

### 1.3 Download Current Data

In [ ]:
# Download current data
current_data = hycom.get_ocean_currents(
    lat_min, lat_max, lon_min, lon_max,
    start_date, end_date
)

print(f'Current data shape: {current_data.dims}')
print(f'Variables: {list(current_data.data_vars)}')

## 2. Wind Analysis

In [ ]:
# Initialize analyzer
wind_analyzer = WindAnalyzer(wind_data)

# Compute magnitude and direction
wind_magnitude = wind_analyzer.compute_magnitude()
wind_direction = wind_analyzer.compute_direction()

# Compute statistics
wind_stats = wind_analyzer.compute_statistics(dim='time')

print(f'Mean wind speed: {wind_stats["mean"].mean().values:.2f} m/s')
print(f'Max wind speed: {wind_stats["max"].max().values:.2f} m/s')
print(f'Min wind speed: {wind_stats["min"].min().values:.2f} m/s')

## 3. Current Analysis

In [ ]:
# Initialize analyzer
current_analyzer = CurrentAnalyzer(current_data, u_var='water_u', v_var='water_v')

# Compute magnitude
current_magnitude = current_analyzer.compute_magnitude(depth_level=0)

# Compute statistics
current_stats = current_analyzer.compute_statistics(depth_level=0, dim='time')

print(f'Mean current speed: {current_stats["mean"].mean().values:.3f} m/s')
print(f'Max current speed: {current_stats["max"].max().values:.3f} m/s')

## 4. Visualization

### 4.1 Wind Field Quiver Plot

In [ ]:
# Create plotter
plotter = QuiverPlotter(figsize=(12, 8), dpi=100)

# Plot wind field
fig, ax = plotter.plot_wind_field(
    wind_data['eastward_wind'],
    wind_data['northward_wind'],
    title='Wind Field - Gulf of Mexico',
    time_index=0,
    subsample=2
)

plt.show()

### 4.2 Ocean Current Field

In [ ]:
# Plot current field
fig, ax = plotter.plot_current_field(
    current_data['water_u'],
    current_data['water_v'],
    title='Ocean Currents - Gulf of Mexico',
    time_index=0,
    depth_index=0,
    subsample=2
)

plt.show()

### 4.3 Combined Wind and Current

In [ ]:
# Combined plot
fig, axes = plotter.plot_combined_wind_current(
    wind_data['eastward_wind'],
    wind_data['northward_wind'],
    current_data['water_u'],
    current_data['water_v'],
    title='Wind and Ocean Currents - Gulf of Mexico',
    subsample=2
)

plt.show()

## 5. Particle Tracking Simulation

### 5.1 Initialize Tracker

In [ ]:
# Initialize particle tracker with RK4 integration
tracker = ParticleTracker(
    current_u=current_data['water_u'],
    current_v=current_data['water_v'],
    wind_u=wind_data['eastward_wind'],
    wind_v=wind_data['northward_wind'],
    wind_drift_coefficient=0.03,
    integration_method='rk4'
)

print('✓ Particle tracker initialized with RK4 integration')

### 5.2 Create Initial Particle Positions

In [ ]:
# Create a grid of particles
initial_positions = tracker.create_particle_grid(
    lon_min=-92.0,
    lon_max=-88.0,
    lat_min=23.0,
    lat_max=27.0,
    n_lon=6,
    n_lat=6
)

print(f'Created {len(initial_positions)} particles')

### 5.3 Run Simulation

In [ ]:
# Run particle tracking
n_steps = 50
dt = 3600 * 3  # 3 hours

trajectories = tracker.track_particles(
    initial_positions=initial_positions,
    n_steps=n_steps,
    dt=dt
)

print(f'Simulation complete!')
print(f'Trajectories shape: {trajectories.shape}')

### 5.4 Visualize Trajectories

In [ ]:
# Plot trajectories
fig, ax = plt.subplots(figsize=(12, 10))

# Plot all trajectories
for i in range(len(initial_positions)):
    traj = trajectories[i]
    valid = ~np.isnan(traj[:, 0])
    if valid.sum() > 1:
        ax.plot(traj[valid, 0], traj[valid, 1], 'b-', alpha=0.5, linewidth=1)

# Plot initial positions
ax.scatter(
    initial_positions[:, 0],
    initial_positions[:, 1],
    c='green', s=100, marker='o',
    label='Initial', zorder=5
)

# Plot final positions
final_positions = trajectories[:, -1, :]
valid_final = ~np.isnan(final_positions[:, 0])
ax.scatter(
    final_positions[valid_final, 0],
    final_positions[valid_final, 1],
    c='red', s=100, marker='x',
    label='Final', zorder=5
)

ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
ax.set_title('Particle Trajectories')
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

## 6. Animation (Optional)

In [ ]:
# Create animator
animator = FieldAnimator(figsize=(12, 8), dpi=80)

# Create wind field animation
try:
    anim = animator.animate_wind_field(
        wind_data['eastward_wind'],
        wind_data['northward_wind'],
        title='Wind Field Animation',
        subsample=3,
        interval=200
    )
    print('Animation created successfully!')
    print('Use animator.save_animation(anim, "output.mp4") to save')
except Exception as e:
    print(f'Animation creation failed: {e}')
    print('This may require FFmpeg or additional dependencies')

## Summary

This notebook demonstrated:

✓ Data acquisition from Copernicus and HYCOM
✓ Wind and current analysis (magnitude, direction, statistics)
✓ Professional quiver plot visualizations
✓ Lagrangian particle tracking simulation
✓ Trajectory visualization

For more examples, see the `examples/` directory.